In [ ]:
# ============================================================
# STEERING TRAINING FOR EMOTION CLASSIFICATION
# ============================================================

import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Imports completed")


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(r"/") #your path
MODEL_DIR = PROJECT_DIR / "models" / "Qwen2.5-Math-1_5B"
TRAIN_PATH = PROJECT_DIR / "emotion_dataset" / "simplified" / "train.parquet"
VALIDATION_PATH = PROJECT_DIR / "emotion_dataset" / "simplified" / "validation.parquet"

OUT_DIR = PROJECT_DIR 
CHECKPOINT_PATH = OUT_DIR / "steering_vectors" / "steering_emotion_qwen.pt"
VALIDATION_STEM = OUT_DIR / "steering_emotion_classification_validation"
XLSX_OUT_PATH = VALIDATION_STEM.with_suffix(".xlsx")
PARQUET_OUT_PATH = VALIDATION_STEM.with_suffix(".parquet")
METRICS_PARQUET_PATH = VALIDATION_STEM.with_name(VALIDATION_STEM.name + "_metrics").with_suffix(".parquet")
PER_CLASS_PARQUET_PATH = VALIDATION_STEM.with_name(VALIDATION_STEM.name + "_per_class").with_suffix(".parquet")
CONFUSION_PARQUET_PATH = VALIDATION_STEM.with_name(VALIDATION_STEM.name + "_confusion_matrix").with_suffix(".parquet")


# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0

NUM_EPOCHS = 3
LR = 2e-4
GRAD_CLIP = 1.0
STEERING_ALPHA = 1.0

MAX_TOTAL_TOKENS = 512
GEN_MAX_NEW_TOKENS = 16
LOG_EVERY = 100
VALIDATION_MAX_ROWS = None

TORCH_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

TEXT_COLUMN = "text"
LABEL_COLUMN = "labels"

GOEMOTIONS_LABELS = {
    0: "admiration", 1: "amusement", 2: "anger", 3: "annoyance",
    4: "approval", 5: "caring", 6: "confusion", 7: "curiosity",
    8: "desire", 9: "disappointment", 10: "disapproval", 11: "disgust",
    12: "embarrassment", 13: "excitement", 14: "fear", 15: "gratitude",
    16: "grief", 17: "joy", 18: "love", 19: "nervousness",
    20: "optimism", 21: "pride", 22: "realization", 23: "relief",
    24: "remorse", 25: "sadness", 26: "surprise", 27: "neutral",
}
CLASS_NAMES = list(GOEMOTIONS_LABELS.values())


# ============================================================
# SEED
# ============================================================

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DATA AND LABEL UTILS
# ============================================================

def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported input format: {suffix}. Use XLSX, XLS, or Parquet.")


def decode_true_label(value):
    # GoEmotions is multi-label. This notebook uses the first label as the
    # single training target, matching the stage 1 inference notebook.
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        if not value:
            return None
        value = value[0]
    if isinstance(value, str):
        value = value.strip()
        if value.startswith("[") or value.startswith("("):
            try:
                return decode_true_label(ast.literal_eval(value))
            except (ValueError, SyntaxError):
                pass
        if value.isdigit():
            return GOEMOTIONS_LABELS.get(int(value))
        return value.lower() or None
    if value is None or pd.isna(value):
        return None
    return GOEMOTIONS_LABELS.get(int(value))


def prepare_dataset(path: Path) -> pd.DataFrame:
    df = load_table(path)
    df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).reset_index(drop=True)
    df["true_label"] = df[LABEL_COLUMN].map(decode_true_label)
    return df.dropna(subset=["true_label"]).reset_index(drop=True)


def parse_predicted_label(text: str) -> str:
    normalized = re.sub(r"[^a-z_ -]+", " ", text.lower()).strip()
    normalized = re.sub(r"\s+", " ", normalized)
    if normalized in CLASS_NAMES:
        return normalized
    matches = []
    for label in CLASS_NAMES:
        for match in re.finditer(rf"(?<!\w){re.escape(label)}(?!\w)", normalized):
            matches.append((match.start(), label))
    return max(matches)[1] if matches else "unknown"


def excel_safe(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value


# ============================================================
# PROMPT
# ============================================================

def make_prompt(text: str) -> str:
    labels = ", ".join(CLASS_NAMES)
    messages = [
        {
            "role": "system",
            "content": (
                "Classify the emotion expressed in the text. "
                f"Choose exactly one label from: {labels}. "
                "Return only the lowercase label and no explanation."
            ),
        },
        {"role": "user", "content": text},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return f"{messages[0]['content']}\n\nTEXT:\n{text}\n\nLABEL:\n"


# ============================================================
# MODEL + TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=TORCH_DTYPE,
    device_map={"": 0} if DEVICE.type == "cuda" else None,
)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad = False

print(f"Model loaded from: {MODEL_DIR}")
print(f"Device: {DEVICE}")


# ============================================================
# STEERING MODULE
# ============================================================

class Steering(nn.Module):
    def __init__(self, model, alpha):
        super().__init__()
        self.layers = model.model.layers
        hidden_size = model.config.hidden_size
        self.vectors = nn.Parameter(
            torch.zeros(len(self.layers), hidden_size, device=DEVICE)
        )
        self.alpha = alpha
        self.handles = []

    def install(self):
        self.remove()

        def make_hook(layer_index):
            def hook(_, __, output):
                if isinstance(output, tuple):
                    output = output[0]
                return output + (self.alpha * self.vectors[layer_index]).to(output.dtype)
            return hook

        for layer_index, layer in enumerate(self.layers):
            handle = layer.mlp.down_proj.register_forward_hook(make_hook(layer_index))
            self.handles.append(handle)

    def remove(self):
        for handle in self.handles:
            try:
                handle.remove()
            except Exception:
                pass
        self.handles = []


steering = Steering(model, STEERING_ALPHA)
steering.install()
optimizer = torch.optim.AdamW([steering.vectors], lr=LR)


# ============================================================
# DATA
# ============================================================

train_df = prepare_dataset(TRAIN_PATH)
validation_df = prepare_dataset(VALIDATION_PATH)
if VALIDATION_MAX_ROWS is not None:
    validation_df = validation_df.head(VALIDATION_MAX_ROWS).copy()

print(f"Loaded training dataset: {len(train_df)} samples")
print(f"Loaded validation dataset: {len(validation_df)} samples")


# ============================================================
# LOSS
# ============================================================

def masked_ce_loss(prompt: str, target: str) -> torch.Tensor:
    target_with_eos = target + tokenizer.eos_token
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        prompt + target_with_eos,
        add_special_tokens=False,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_TOTAL_TOKENS,
    )
    input_ids = full["input_ids"].to(DEVICE)
    attention_mask = full["attention_mask"].to(DEVICE)
    labels = input_ids.clone()
    labels[:, :min(len(prompt_ids), input_ids.shape[1] - 1)] = -100

    steering.alpha = STEERING_ALPHA
    return model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        use_cache=False,
    ).loss


# ============================================================
# GENERATE
# ============================================================

@torch.no_grad()
def generate_label(text: str, alpha: float):
    old_alpha = steering.alpha
    steering.alpha = alpha
    prompt = make_prompt(text)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    seq = model.generate(
        **enc,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_tokens = seq[0, enc["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    predicted_label = parse_predicted_label(raw_output)
    steering.alpha = old_alpha
    return raw_output, predicted_label


# ============================================================
# TRAIN LOOP
# ============================================================

global_step = 0

for epoch in range(1, NUM_EPOCHS + 1):
    shuffled_train_df = train_df.sample(
        frac=1.0, random_state=SEED + epoch
    ).reset_index(drop=True)
    epoch_loss = 0.0
    progress = tqdm(
        shuffled_train_df.iterrows(),
        total=len(shuffled_train_df),
        desc=f"Epoch {epoch}/{NUM_EPOCHS}",
    )

    for _, row in progress:
        global_step += 1
        text = str(row[TEXT_COLUMN])
        target_label = str(row["true_label"])

        loss = masked_ce_loss(make_prompt(text), target_label)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([steering.vectors], GRAD_CLIP)
        optimizer.step()
        epoch_loss += loss.item()

        if global_step % LOG_EVERY == 0:
            progress.set_postfix(
                loss=f"{loss.item():.4f}",
                steering_norm=f"{steering.vectors.norm().item():.4f}",
            )

    average_epoch_loss = epoch_loss / len(shuffled_train_df)
    print(
        f"[EPOCH {epoch}/{NUM_EPOCHS}] "
        f"average_loss={average_epoch_loss:.4f} | "
        f"optimizer_steps={global_step}"
    )


# ============================================================
# VALIDATION: WITHOUT STEERING VS WITH STEERING
# ============================================================

validation_rows = []
for i, row in tqdm(validation_df.iterrows(), total=len(validation_df)):
    text = str(row[TEXT_COLUMN])
    true_label = str(row["true_label"])
    raw_off, predicted_off = generate_label(text, alpha=0.0)
    raw_on, predicted_on = generate_label(text, alpha=STEERING_ALPHA)

    result = row.to_dict()
    result.update({
        "row_id": i,
        "predicted_off": predicted_off,
        "predicted_on": predicted_on,
        "raw_output_off": raw_off,
        "raw_output_on": raw_on,
        "correct_off": true_label == predicted_off,
        "correct_on": true_label == predicted_on,
    })
    validation_rows.append(result)

validation_results_df = pd.DataFrame(validation_rows)
y_true = validation_results_df["true_label"].tolist()


def calculate_metrics(prediction_column: str, mode: str):
    y_pred = validation_results_df[prediction_column].tolist()
    summary = {
        "mode": mode,
        "macro_f1": f1_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0),
        "unknown_predictions": int((validation_results_df[prediction_column] == "unknown").sum()),
        "dataset_size": len(validation_results_df),
    }

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASS_NAMES, zero_division=0
    )
    per_class = pd.DataFrame({
        "mode": mode,
        "label": CLASS_NAMES,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })

    confusion_labels = CLASS_NAMES.copy()
    if "unknown" in y_pred:
        confusion_labels.append("unknown")
    confusion_wide = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=confusion_labels),
        index=confusion_labels,
        columns=confusion_labels,
    ).rename_axis("true_label").reset_index()
    confusion_long = confusion_wide.melt(
        id_vars="true_label",
        var_name="predicted_label",
        value_name="count",
    )
    confusion_long.insert(0, "mode", mode)
    return summary, per_class, confusion_wide, confusion_long


summary_off, per_class_off, confusion_off, confusion_long_off = calculate_metrics(
    "predicted_off", "without_steering"
)
summary_on, per_class_on, confusion_on, confusion_long_on = calculate_metrics(
    "predicted_on", "with_steering"
)
metrics_df = pd.DataFrame([summary_off, summary_on])
per_class_df = pd.concat([per_class_off, per_class_on], ignore_index=True)
confusion_long_df = pd.concat([confusion_long_off, confusion_long_on], ignore_index=True)

print("\nValidation metrics:")
print(metrics_df.to_string(index=False))
print(f"Macro F1 change: {summary_on['macro_f1'] - summary_off['macro_f1']:+.4f}")
print("\nConfusion matrix without steering:")
print(confusion_off.to_string(index=False))
print("\nConfusion matrix with steering:")
print(confusion_on.to_string(index=False))


# ============================================================
# SAVE CHECKPOINT + XLSX + PARQUET
# ============================================================

OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "vectors": steering.vectors.detach().float().cpu(),
        "alpha": STEERING_ALPHA,
        "epochs": NUM_EPOCHS,
        "optimizer_steps": global_step,
        "model_dir": str(MODEL_DIR),
        "class_names": CLASS_NAMES,
        "label_mapping": GOEMOTIONS_LABELS,
    },
    CHECKPOINT_PATH,
)

excel_validation = validation_results_df.copy()
for column in excel_validation.columns:
    excel_validation[column] = excel_validation[column].map(excel_safe)

with pd.ExcelWriter(XLSX_OUT_PATH) as writer:
    excel_validation.to_excel(writer, sheet_name="validation_predictions", index=False)
    metrics_df.to_excel(writer, sheet_name="summary", index=False)
    per_class_df.to_excel(writer, sheet_name="per_class", index=False)
    confusion_off.to_excel(writer, sheet_name="confusion_off", index=False)
    confusion_on.to_excel(writer, sheet_name="confusion_on", index=False)

validation_results_df.to_parquet(PARQUET_OUT_PATH, index=False)
metrics_df.to_parquet(METRICS_PARQUET_PATH, index=False)
per_class_df.to_parquet(PER_CLASS_PARQUET_PATH, index=False)
confusion_long_df.to_parquet(CONFUSION_PARQUET_PATH, index=False)

print("\nTraining finished.")
print("Saved steering checkpoint to:", CHECKPOINT_PATH)
print("Saved XLSX validation results to:", XLSX_OUT_PATH)
print("Saved Parquet validation predictions to:", PARQUET_OUT_PATH)
print("Saved Parquet metrics to:", METRICS_PARQUET_PATH)
print("Saved Parquet per-class metrics to:", PER_CLASS_PARQUET_PATH)
print("Saved Parquet confusion matrix to:", CONFUSION_PARQUET_PATH)
